# WavLM Supervised Contrastive Fine-tuning on AbjadKids

End-to-end notebook reproducing the experiment described in this repository. Designed to run on Google Colab with a T4 GPU. All numeric results in `docs/results.md` were obtained with this configuration.

## Setup

In [ ]:
!pip install -q torch torchaudio transformers==4.37.2 tokenizers==0.15.2 huggingface-hub==0.20.3 datasets hf_transfer accelerate soundfile librosa scikit-learn hdbscan tqdm
!pip install --force-reinstall --no-cache-dir 'numpy==1.26.4'

In [ ]:
import os

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_DATASETS_CACHE'] = '/content/hf_datasets_cache'

In [ ]:
from huggingface_hub import login
login()  # paste your HF token when prompted

## Download AbjadKids and Build Splits

In [ ]:
from huggingface_hub import snapshot_download

DATA_DIR = '/content/AbjadKids'
snapshot_download(
    repo_id='Aziz-snoubra/Abjad-Kids',
    repo_type='dataset',
    local_dir=DATA_DIR,
    local_dir_use_symlinks=False,
    resume_download=True,
    max_workers=16,
)

In [ ]:
from src.dataset import build_splits, index_dataset

df = index_dataset(
    data_dir=DATA_DIR,
    categories=['alphabet', 'colors', 'numbers'],
    audio_extensions=['.wav', '.mp3', '.flac', '.ogg', '.m4a'],
)
splits = build_splits(df, excluded_classes=['Walad'])
train_df, val_df, test_df = splits['train'], splits['val'], splits['test']
label2id, id2label = splits['label2id'], splits['id2label']
print(len(train_df), len(val_df), len(test_df))

## Train

In [ ]:
from src.train import TrainConfig, train

cfg = TrainConfig(checkpoint_dir='/content/checkpoints')
model = train(train_df, label2id, id2label, cfg, device='cuda')

## Evaluate

In [ ]:
from src.evaluate import evaluate_checkpoint

results = evaluate_checkpoint(
    checkpoint_path='/content/checkpoints/wavlm_supcon_step_2000.pt',
    test_df=test_df,
)
for key, value in results.items():
    print(f'{key}: {value}')